In [ ]:
import serial
import serial.tools.list_ports
import time
import numpy as np
import math
import os
from datetime import datetime
import threading
import tkinter as tk
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

import traceback
from threading import current_thread
from tkinter import messagebox
import random


DEBUG = False
def dbg(tag, **kv):
    if not DEBUG: return
    t = time.strftime("%H:%M:%S")
    kvs = " ".join(f"{k}={v}" for k,v in kv.items())
    print(f"[{t}] [{tag}] {kvs}")

BIT_DUMP_DIR = "bit_dumps"   # mappe til 0/1-filer
os.makedirs(BIT_DUMP_DIR, exist_ok=True)

# ──────────────────────────────────────────────────────────────────────────────
# Konfiguration
VIDPID             = "16C0:0483"   # Teensy 4.1 VID:PID
BAUDRATE           = 9600
TIMEOUT            = 5
SAMPLES_PER_SECOND = 15000
BLOCK_SIZE         = 1500
LOG_FILENAME = "accum_log-no_lowpass-LSB-von-neumann.csv"   # gemmer også bits/raw_samples
DOWNSAMPLE         = 1
MASK_BITS          = 0x3

# Tråd-/state-objekter
stop_event = threading.Event()
ser_lock = threading.Lock()
current_worker = None

# Global toggle husker fasen mellem runs
global_toggle = 0

# ── Digitalt lavpas til decimering ───────────────────────────────────
USE_DECIM_LPF  = False
DECIM_TAPS     = 511
DECIM_WINDOW   = "blackman"
PASS_FRAC      = 0.9
ADC_BITS       = 12
ADC_MAX        = (1 << ADC_BITS) - 1
ADC_MID        = ADC_MAX // 2

def design_lowpass_taps(num_taps, D, window="blackman", pass_frac=0.9):
    fc_nyq = pass_frac / float(D)
    fc_fs  = fc_nyq / 2.0
    n = np.arange(num_taps) - (num_taps - 1) / 2.0
    h = 2.0 * fc_fs * np.sinc(2.0 * fc_fs * n)
    wname = (window or "").lower()
    if   wname == "blackman": w = np.blackman(num_taps)
    elif wname == "hann":     w = np.hanning(num_taps)
    elif wname == "hamming":  w = np.hamming(num_taps)
    else:                     w = np.ones(num_taps)
    h *= w
    h /= h.sum() + 1e-20
    return h.astype(np.float64)

def _report_fc(h, fs_hz, label="LPF"):
    N = 1 << 15
    H = np.fft.rfft(h, N)
    f = np.fft.rfftfreq(N, 1.0/fs_hz)
    mag = 20.0*np.log10(np.abs(H) / (np.abs(H[0]) + 1e-20) + 1e-20)
    idx = np.argmax(mag <= -3.0)
    fc_meas = f[idx] if idx > 0 else float('nan')
    print(f"{label}: målt -3 dB ≈ {fc_meas:.1f} Hz")

# ──────────────────────────────────────────────────────────────────────────────
# Statistik-hjælpere
def z_from_accum(accum, N_bits):
    if N_bits <= 0: return float("nan")
    sigma = math.sqrt(N_bits)
    return accum / sigma if sigma > 0 else float("nan")


def p_two_sided_from_z(z):
    if not math.isfinite(z): return float("nan")
    return math.erfc(abs(z)/math.sqrt(2.0))

def p_one_sided_up(z):
    if not math.isfinite(z): return float("nan")
    return 0.5 * math.erfc(z / math.sqrt(2.0))   # = 1-Φ(z)

def p_one_sided_down(z):
    if not math.isfinite(z): return float("nan")
    return 0.5 * math.erfc(-z / math.sqrt(2.0))  # = Φ(z)

def norm_ppf(p):
    if not (0.0 < p < 1.0): return float("nan")
    a = [-3.969683028665376e+01,  2.209460984245205e+02,
         -2.759285104469687e+02,  1.383577518672690e+02,
         -3.066479806614716e+01,  2.506628277459239e+00]
    b = [-5.447609879822406e+01,  1.615858368580409e+02,
         -1.556989798598866e+02,  6.680131188771972e+01,
         -1.328068155288572e+01]
    c = [-7.784894002430293e-03, -3.223964580411365e-01,
         -2.400758277161838e+00, -2.549732539343734e+00,
          4.374664141464968e+00,  2.938163982698783e+00]
    d = [ 7.784695709041462e-03,  3.224671290700398e-01,
          2.445134137142996e+00,  3.754408661907416e+00]
    plow  = 0.02425
    phigh = 1 - plow
    if p < plow:
        q = math.sqrt(-2*math.log(p))
        return (((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
               ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
    if p > phigh:
        q = math.sqrt(-2*math.log(1-p))
        return -(((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                 ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
    q = p - 0.5
    r = q*q
    return (((((a[0]*r + a[1])*r + a[2])*r + a[3])*r + a[4])*r + a[5])*q / \
           (((((b[0]*r + b[1])*r + b[2])*r + b[3])*r + b[4])*r + 1)

def fisher_p_from_two_sided(p1, p2):
    if not (0 < p1 < 1 and 0 < p2 < 1):
        return float("nan")
    T = -2.0 * (math.log(p1) + math.log(p2))
    return math.exp(-T/2.0) * (1.0 + T/2.0)  # Chi2(4) sf

def z_equiv_from_two_sided_p(p_two):
    if not (0 < p_two < 1): return float("nan")
    q = 1.0 - p_two/2.0
    return abs(norm_ppf(q))

def stouffer_combined(z_up, n_up, z_down, n_down, directional=True):
    if not (math.isfinite(z_up) and math.isfinite(z_down)):
        return float("nan"), float("nan"), float("nan")
    if n_up <= 0 or n_down <= 0:
        return float("nan"), float("nan"), float("nan")
    w_up = math.sqrt(n_up)
    w_dn = math.sqrt(n_down)
    z_dn_eff = -z_down if directional else z_down
    denom = math.sqrt(w_up*w_up + w_dn*w_dn)
    if denom == 0:
        return float("nan"), float("nan"), float("nan")
    zc = (w_up*z_up + w_dn*z_dn_eff) / denom
    p1 = 0.5 * math.erfc(zc / math.sqrt(2.0))
    p2 = math.erfc(abs(zc)/math.sqrt(2.0))
    return zc, p1, p2

# ──────────────────────────────────────────────────────────────────────────────
# Serial-hjælpere
def find_port(vidpid=VIDPID):
    for p in serial.tools.list_ports.comports():
        if vidpid in p.hwid:
            return p.device
    raise IOError("Teensy ikke fundet på portene")

def open_serial(baudrate=BAUDRATE):
    while True:
        try:
            port = find_port()
            ser = serial.Serial(port, baudrate=baudrate, timeout=TIMEOUT)
            ser.dtr = False
            ser.reset_input_buffer()
            dbg("SER_OPEN", port=port, baudrate=baudrate)
            return ser
        except IOError:
            time.sleep(1)

def sync(ser, overall=2.0):
    start = time.monotonic()
    while not stop_event.is_set():
        if time.monotonic() - start > overall:
            dbg("SYNC_TIMEOUT")
            raise RuntimeError("SYNC_TIMEOUT")
        try:
            b = ser.read(1)
        except (serial.SerialException, AttributeError):
            raise InterruptedError("Serial closed during sync()")
        if not b:
            continue
        if b == b'\x7E':
            dbg("SYNC_OK")
            return
    raise InterruptedError("Stop requested during sync()")

def read_block(ser):
    sync(ser)
    try:
        hdr = ser.read(7)
    except (serial.SerialException, AttributeError):
        raise InterruptedError("Serial closed during header")
    if stop_event.is_set():
        raise InterruptedError("Stop requested during header")
    if len(hdr) < 7:
        raise RuntimeError("Timeout på header")

    adc, hi, lo, b3, b2, b1, b0 = hdr
    count = (hi << 8) | lo
    block_id = (b3 << 24) | (b2 << 16) | (b1 << 8) | b0

    dbg("HDR", adc=adc, count=count, block_id=block_id)

    need = count * 2
    data = bytearray()
    while len(data) < need and not stop_event.is_set():
        try:
            chunk = ser.read(need - len(data))
        except (serial.SerialException, AttributeError):
            raise InterruptedError("Serial closed during data")
        if not chunk:
            continue
        data.extend(chunk)

    if stop_event.is_set():
        raise InterruptedError("Stop requested during data")
    if len(data) < need:
        raise RuntimeError("Timeout på data")

    dbg("DATA_OK", words=len(data)//2, block_id=block_id)
    return adc, block_id, np.frombuffer(bytes(data), dtype=np.uint16)

# ──────────────────────────────────────────────────────────────────────────────
# Hovedmåling
def record_samples(ser, total_samples=150000, return_bits=False, toggle_start=None, on_progress=None, target_vn_bits=None):

    if toggle_start is None:
        toggle = 0
    else:
        toggle = toggle_start
    
    if USE_DECIM_LPF:
        print(f"Bruger hver {DOWNSAMPLE}. sample som bit (med anti-alias LPF)")
    else:
        print(f"Bruger hver {DOWNSAMPLE}. sample som bit (uden LPF)")


    adc0_blocks = 0
    adc1_blocks = 0
    accum    = 0
    raw_cnt  = 0
    bit_cnt  = 0
    vn_pending = None
    reached_target = False
    
    bits_out = [] if return_bits else None  # samler 0/1 til dump

    last_block_id = None
    gap_count = 0

    blocks_needed = (total_samples + BLOCK_SIZE - 1) // BLOCK_SIZE
    try:
        with ser_lock:
            if stop_event.is_set():
                if return_bits:
                    return 0.0, 0, 0, "", toggle
                else:
                    return 0.0, 0, 0, toggle
            
            dbg("SEND_START", blocks=blocks_needed)
            ser.write(f"START {blocks_needed}\n".encode())
            ser.flush()
            
        time.sleep(0.12)  # ← kort settle inden sync/read
        
    except Exception:
        if return_bits:
            return 0.0, 0, 0, "", toggle
        else:
            return 0.0, 0, 0, toggle

    if USE_DECIM_LPF:
        lpf_h   = design_lowpass_taps(DECIM_TAPS, DOWNSAMPLE, DECIM_WINDOW, PASS_FRAC)
        from collections import deque
        lpf_buf = deque(maxlen=DECIM_TAPS)
        fc_nom = PASS_FRAC * (SAMPLES_PER_SECOND / (2.0 * DOWNSAMPLE))
        print(f"Decim-LPF: taps={DECIM_TAPS}, window={DECIM_WINDOW}, nominel cutoff ≈ {fc_nom:.1f} Hz")
        _report_fc(lpf_h, SAMPLES_PER_SECOND, "Decim-LPF")

    try:
        retry_done = False

        while raw_cnt < total_samples and not stop_event.is_set():
            try:
                adc, block_id, block = read_block(ser)
                if adc == 0:
                    adc0_blocks += 1
                elif adc == 1:
                    adc1_blocks += 1
                if adc == 0:
                    if last_block_id is not None:
                        expected = (last_block_id + 1) & 0xFFFFFFFF
                        if block_id != expected:
                            gap = block_id - expected
                            gap_count += 1
                            print(f"⚠️ Manglende blokke på ADC0: forventet {expected}, fik {block_id} (gap={gap})")
                    last_block_id = block_id
            except RuntimeError as e:  # SYNC_TIMEOUT / header/data timeout
                if not retry_done:
                    dbg("RETRY_AFTER_TIMEOUT", err=str(e))
                    with ser_lock:
                        ser.reset_input_buffer()
                        ser.reset_output_buffer()
                        ser.write(f"START {blocks_needed}\n".encode())
                        ser.flush()
                    time.sleep(0.12)
                    retry_done = True
                    continue
                else:
                    dbg("ABORT_AFTER_TIMEOUT", err=str(e))
                    break
            except InterruptedError:
                break
            except (serial.SerialException, AttributeError):
                break

            if adc != 0:
                continue

            needed = total_samples - raw_cnt
            raw = block if len(block) <= needed else block[:needed]

            
            for raw_val in raw:
                if stop_event.is_set():
                    break

                raw_cnt += 1

                if USE_DECIM_LPF:
                    x = float(int(raw_val) - ADC_MID)
                    lpf_buf.append(x)
                    if len(lpf_buf) < DECIM_TAPS:
                        continue
                    if (raw_cnt % DOWNSAMPLE) != 0:
                        continue
                    buf = np.fromiter(lpf_buf, dtype=np.float64, count=DECIM_TAPS)
                    y   = float(np.dot(lpf_h, buf))
                    val_for_bits = int(np.clip(y + ADC_MID, 0, ADC_MAX))
                else:
                    if (raw_cnt % DOWNSAMPLE) != 0:
                        continue
                    val_for_bits = int(raw_val)

                b    = val_for_bits & MASK_BITS
                bit0 = b & 1
                # --- Von Neumann på bit0-strømmen ---
                if vn_pending is None:
                    vn_pending = bit0
                    continue
                
                a = vn_pending
                b = bit0
                vn_pending = None
                
                if a == b:
                    continue  # 00 eller 11 -> kassér
                
                # 01 -> 0, 10 -> 1
                vn_bit = 1 if (a == 1 and b == 0) else 0
                
                accum += (1 if vn_bit else -1)
                bit_cnt += 1

                if return_bits:
                    bits_out.append('1' if vn_bit else '0')                
                
                if target_vn_bits is not None and bit_cnt >= target_vn_bits:
                    reached_target = True
                    break
                
                if bit_cnt % 1000 == 0:
                    if on_progress is not None:
                        on_progress(accum, bit_cnt, raw_cnt)

            if reached_target:
                break

                #if bit_cnt >= total_samples // DOWNSAMPLE:
                #    break

            #if bit_cnt >= total_samples // DOWNSAMPLE:
            #    break

    finally:
        try:
            with ser_lock:
                if ser and ser.is_open:
                    dbg("SEND_STOP")
                    ser.write(b"STOP\n")   # ← newline
                    ser.flush()
                    time.sleep(0.05)       # ← lille settle
                    ser.reset_input_buffer()
                    ser.reset_output_buffer()
        except Exception as e:
            dbg("STOP_ERR", err=repr(e))

    print("\nBlokke modtaget i dette run:")
    print(f"  ADC0: {adc0_blocks}")
    print(f"  ADC1: {adc1_blocks}")
    
    if gap_count == 0:
        print("Ingen manglende ADC0-blokke registreret i dette run.")
    else:
        print(f"Registrerede {gap_count} blok-gap i dette run.")
    if return_bits:
        bits_str = "".join(bits_out)
        return accum, bit_cnt, raw_cnt, bits_str, toggle
    else:
        return accum, bit_cnt, raw_cnt, toggle


# ──────────────────────────────────────────────────────────────────────────────
# GUI og hovedprogram
def main():
    global current_worker

    ser = open_serial()
    print(f">> Forbundet til Teensy på {ser.port}")

    # Debug/tilstand
    last_click_time = 0.0
    last_start_time = 0.0
    last_done_time  = 0.0
    run_seq = 0

    run_target_raw = 0

    run_start_monotonic = None

    active_run_id = 0   # token for hvilket run der er aktivt (live)

    last_err = None
    MIN_CLICK_GAP = 0.35  # sekunder, debounce mod "hurtige klik"

    
    # Historik og kumulative tællere
    accum_up = 0
    accum_down = 0
    accum_none = 0
    
    cum_up_bits = 0
    cum_down_bits = 0
    cum_none_bits = 0

    cum_up_raw = 0
    cum_down_raw = 0
    cum_none_raw = 0
    
    history_up_bits   = [0]
    history_up        = [0]
    
    history_down_bits = [0]
    history_down      = [0]
    
    history_none_bits = [0]
    history_none      = [0]
    

    # NEW: last-run state (retning, delta, bits)
    last_run_dir  = None
    last_run_val  = None
    last_run_bits = None
    last_run_mode = None

    # Læs eksisterende log (bagud-kompatibel)
    if os.path.exists(LOG_FILENAME):
        with open(LOG_FILENAME, 'r', encoding="utf-8") as f:
            header = next(f, "")
            for line in f:
                parts = line.strip().split(',')
                if len(parts) == 6:
                    ts, mode, direction, total, bits_str, raw_str = parts

                    bits = int(bits_str)
                    raws = int(raw_str)
                    
                    if direction == 'UP':
                        accum_up = int(float(total))
                        cum_up_bits += bits
                        cum_up_raw += raws
                        history_up.append(accum_up)
                        history_up_bits.append(cum_up_bits)
                    
                    elif direction == 'DOWN':
                        accum_down = int(float(total))
                        cum_down_bits += bits
                        cum_down_raw += raws
                        history_down.append(accum_down)
                        history_down_bits.append(cum_down_bits)
                    
                    elif direction in ('NONE', 'BASE', 'NEUTRAL'):
                        accum_none = int(float(total))
                        cum_none_bits += bits
                        cum_none_raw += raws
                        history_none.append(accum_none)
                        history_none_bits.append(cum_none_bits)


    # Åbn log (med header hvis nødvendig)
    need_header = not os.path.exists(LOG_FILENAME) or os.path.getsize(LOG_FILENAME) == 0 or \
        (open(LOG_FILENAME, "r", encoding="utf-8").readline().strip() != "timestamp,mode,direction,total,bits,raw_samples")
    log_file = open(LOG_FILENAME, "a", encoding="utf-8")
    if need_header:
        log_file.write("timestamp,mode,direction,total,bits,raw_samples\n")
        log_file.flush()

    # GUI
    root = tk.Tk()
    root.title("Teensy Quantum Noise Monitor (LSB + Von Neumann)")

    title = tk.Label(root, text="Teensy Quantum Noise Monitor (LSB + Von Neumann)", font=("Segoe UI", 16, "bold"))
    title.pack(pady=(6,4))

    grid = tk.Frame(root)

    grid.grid_columnconfigure(0, weight=0, minsize=90)    # Channel
    grid.grid_columnconfigure(1, weight=0, minsize=110)   # Accumulator
    grid.grid_columnconfigure(2, weight=0, minsize=170)   # Samples / Bits  (var 260)
    grid.grid_columnconfigure(3, weight=1, minsize=160)   # p-values / Z    (var 220)


    
    grid.pack(padx=8, pady=6, fill=tk.X)

    tk.Label(grid, text="Channel",        font=("Segoe UI", 11, "bold")).grid(row=0, column=0, sticky="w")
    tk.Label(grid, text="Accumulator",    font=("Segoe UI", 11, "bold")).grid(row=0, column=1, sticky="w")
    tk.Label(grid, text="Samples / Bits", font=("Segoe UI", 11, "bold")).grid(row=0, column=2, sticky="w")
    tk.Label(grid, text="p-values / Z",   font=("Segoe UI", 11, "bold")).grid(row=0, column=3, sticky="w")

    tk.Label(grid, text="UP",     fg="#2b7e2b", font=("Segoe UI", 11, "bold")).grid(row=1, column=0, sticky="w")
    tk.Label(grid, text="DOWN",   fg="#1f57b5", font=("Segoe UI", 11, "bold")).grid(row=2, column=0, sticky="w")
    tk.Label(grid, text="NONE", fg="#595959", font=("Segoe UI", 11, "bold")).grid(row=3, column=0, sticky="w")
    
    tk.Label(
        grid,
        text="COMBINED (Fisher & Stouffer)",
        font=("Segoe UI", 11, "bold"),
        justify="left"
    ).grid(row=4, column=0, sticky="w")


    up_acc_label   = tk.Label(grid, text=f"{accum_up:+d}",   font=("Consolas", 11))
    down_acc_label = tk.Label(grid, text=f"{accum_down:+d}", font=("Consolas", 11))
    none_acc_label = tk.Label(grid, text=f"{accum_none:+d}", font=("Consolas", 11))
    
    up_acc_label.grid(row=1, column=1, sticky="w")
    down_acc_label.grid(row=2, column=1, sticky="w")
    none_acc_label.grid(row=3, column=1, sticky="w")


    up_samples_label   = tk.Label(grid, text=f"{cum_up_raw:,} rå samples  |  {cum_up_bits:,} bits", font=("Consolas", 11))
    down_samples_label = tk.Label(grid, text=f"{cum_down_raw:,} rå samples  |  {cum_down_bits:,} bits", font=("Consolas", 11))
    none_samples_label = tk.Label(grid, text=f"{cum_none_raw:,} rå samples  |  {cum_none_bits:,} bits", font=("Consolas", 11))
    
    up_samples_label.grid(row=1, column=2, sticky="w")
    down_samples_label.grid(row=2, column=2, sticky="w")
    none_samples_label.grid(row=3, column=2, sticky="w")


    p_up_label     = tk.Label(grid, text="", font=("Consolas", 11))
    p_down_label   = tk.Label(grid, text="", font=("Consolas", 11))
    p_none_label   = tk.Label(grid, text="", font=("Consolas", 11))
    p_comb_label   = tk.Label(grid, text="", font=("Consolas", 11))
    p_last_label   = tk.Label(grid, text="", font=("Consolas", 11))
    
    live_run_label = tk.Label(
        grid,
        text="",
        font=("Consolas", 11),
        anchor="w",
        justify="left",
        width=160
    )
    
    p_up_label.grid(row=1, column=3, sticky="w")
    p_down_label.grid(row=2, column=3, sticky="w")
    p_none_label.grid(row=3, column=3, sticky="w")
    
    p_comb_label.grid(row=4, column=1, columnspan=3, sticky="w")
    p_last_label.grid(row=5, column=1, columnspan=3, sticky="w")
    live_run_label.grid(row=6, column=1, columnspan=3, sticky="w")
    
    tk.Label(grid, text="LAST RUN", font=("Segoe UI", 11, "bold")).grid(row=5, column=0, sticky="w")
    tk.Label(grid, text="RUN (live)", font=("Segoe UI", 11, "bold")).grid(row=6, column=0, sticky="w")


    # Plot
    fig = Figure(figsize=(6.4,3.6))
    ax = fig.add_subplot(111)

    ax.set_title("LSB Sampling + Von Neumann Debiasing")
    
    ax.set_xlabel("Faktiske bits")
    ax.set_ylabel("Accumulator")
    
    (line_up,)   = ax.plot(history_up_bits,   history_up,   color="#2b7e2b", lw=1.8, label="UP")
    (line_down,) = ax.plot(history_down_bits, history_down, color="#1f57b5", lw=1.8, label="DOWN")
    (line_none,) = ax.plot(history_none_bits, history_none, color="0.35",    lw=1.4, label="NONE")

    # live-kurve for igangværende run
    (line_live,) = ax.plot([], [], color="tab:orange", lw=1.3, linestyle="--", label="RUN live")



    ax.legend(ncol=2)

    sigma_artists = []

    # ---- Sigma-linjer: opret én gang, opdatér data efter behov ----
    sigma_lines = {}  # fx {"1p": line, "1m": line, ...}
    
    def init_sigma_lines():
        nonlocal sigma_lines
        # 6 linjer: ±1σ, ±2σ, ±3σ
        # (du kan beholde dine farver/stilarter)
        
        sigma_lines["1p"], = ax.plot([], [], '-',  color='0.35', lw=1.8, label='+1σ')
        sigma_lines["1m"], = ax.plot([], [], '-',  color='0.35', lw=1.8, label='−1σ')
        sigma_lines["2p"], = ax.plot([], [], '--', color='0.6',  lw=1.0, label='+2σ')
        sigma_lines["2m"], = ax.plot([], [], '--', color='0.6',  lw=1.0, label='−2σ')
        sigma_lines["3p"], = ax.plot([], [], '-',  color='tab:olive', lw=2.0, label='+3σ')
        sigma_lines["3m"], = ax.plot([], [], '-',  color='tab:olive', lw=2.0, label='−3σ')
        
    def update_sigma_lines(max_bits):
        max_bits = max(1, int(max_bits))
        
        # Flere punkter nær 0 (gør kurverne glatte)
        t = np.linspace(0.0, math.sqrt(max_bits), 1200)   # 1200 kan justeres
        xs = t * t
        base = 1.0 * t   # fordi sqrt(xs) = t
    
        sigma_lines["1p"].set_data(xs, +1*base)
        sigma_lines["1m"].set_data(xs, -1*base)
        sigma_lines["2p"].set_data(xs, +2*base)
        sigma_lines["2m"].set_data(xs, -2*base)
        sigma_lines["3p"].set_data(xs, +3*base)
        sigma_lines["3m"].set_data(xs, -3*base)
    
        # sørg for at de altid er synlige
        #for ln in sigma_lines.values():
        #    ln.set_visible(True)
    init_sigma_lines()
    
    def refresh_legend():
        handles, labels = ax.get_legend_handles_labels()
    
        seen = set()
        uh, ul = [], []
        for h, l in zip(handles, labels):
            if not l or l.startswith("_"):
                continue
            if l in seen:
                continue
            seen.add(l)
            uh.append(h)
            ul.append(l)
    
        #ax.legend(uh, ul, ncol=2)
        ax.legend(uh, ul, ncol=2, loc="upper left", framealpha=0.5, facecolor="white")

    
    def enforce_sigma_visibility():
        # ±1σ ALDRIG skjult
        sigma_lines["1p"].set_visible(True)
        sigma_lines["1m"].set_visible(True)
    
        # ±2σ og ±3σ: synlige, men må ryge ud af y-limits
        for k in ("2p","2m","3p","3m"):
            sigma_lines[k].set_visible(True)

    # ⬇️ LAV LIVE-LISTER HER
    live_x = []
    live_y = []

    live_mode = False
    run_target_bits = 1


    def refresh_live_plot():
        line_live.set_xdata(live_x)
        line_live.set_ydata(live_y)
    
        if live_mode:
            # x: hele run'et fra start
            ax.set_xlim(0, max(1, run_target_bits))
    
            # 1σ-ramme for hele run'et (så den fylder billedet fra start)
            sigma1 = math.sqrt(max(1, run_target_bits))
    
            if live_y:
                y_min = min(live_y)
                y_max = max(live_y)
                y_abs = max(abs(y_min), abs(y_max))
    
                # hold mindst 1σ, men udvid hvis data går udenfor
                y_lim = max(sigma1, 1.10 * y_abs, 1.0)
                ax.set_ylim(-y_lim, y_lim)
            else:
                # helt i starten (ingen data endnu): ±1σ fylder hele billedet
                ax.set_ylim(-sigma1, sigma1)
    
            # sigma-kurver i live view skal følge run_target_bits (ikke historik)
    
        canvas.draw_idle()

    


    
    def fmt_mmss(seconds: float) -> str:
        if seconds is None or seconds < 0 or not math.isfinite(seconds):
            return "--:--"
        s = int(seconds + 0.5)
        m = s // 60
        s = s % 60
        return f"{m:02d}:{s:02d}"
    
    def update_live_run(direction, accum_val, bit_cnt, raw_used, run_id):
        nonlocal active_run_id, live_mode, run_start_monotonic, run_target_bits, run_target_raw
        if (not live_mode) or (run_id != active_run_id):
            return
    
        # Statistik
        z_live = z_from_accum(accum_val, bit_cnt)
        p_live_2 = p_two_sided_from_z(z_live)
        if direction == "UP":
            p_live_1 = p_one_sided_up(z_live)
            p_txt = f"p≈{p_live_1:.3g} (1s) / {p_live_2:.3g} (2s)"
        elif direction == "DOWN":
            p_live_1 = p_one_sided_down(z_live)
            p_txt = f"p≈{p_live_1:.3g} (1s) / {p_live_2:.3g} (2s)"
        else:
            p_txt = f"p≈{p_live_2:.3g} (2s)"
    
        # Tid og hastigheder
        now = time.monotonic()
        elapsed = (now - run_start_monotonic) if run_start_monotonic else float("nan")
        bits_per_s = (bit_cnt / elapsed) if (elapsed > 0 and math.isfinite(elapsed)) else float("nan")
        raw_per_s  = (raw_used / elapsed) if (elapsed > 0 and math.isfinite(elapsed)) else float("nan")
    
        # Progress + ETA
        frac = (bit_cnt / run_target_bits) if run_target_bits > 0 else 0.0
        eta = ((run_target_bits - bit_cnt) / bits_per_s) if (bits_per_s > 0 and math.isfinite(bits_per_s)) else float("nan")
    
        live_run_label.config(
            text=(
                f"RUN={direction}  "
                f"raw={raw_used:,}/{run_target_raw:,}  bits={bit_cnt:,}/{run_target_bits:,}  "
                f"({frac*100:5.1f}% )\n"
                f"t={fmt_mmss(elapsed)}  ETA={fmt_mmss(eta)}  "
                f"rate≈{raw_per_s:,.0f} raw/s  {bits_per_s:,.0f} bits/s  |  "
                f"Δ={accum_val:+.0f}  Z={z_live:+.2f}  {p_txt}"
            )
        )
    
        live_x.append(bit_cnt)
        live_y.append(accum_val)
        refresh_live_plot()



    canvas = FigureCanvasTkAgg(fig, master=root)
    widget = canvas.get_tk_widget()
    widget.pack(fill=tk.BOTH, expand=True)

    def max_history_bits():
        return max(history_up_bits[-1] if history_up_bits else 0,
                   history_down_bits[-1] if history_down_bits else 0,
                   history_none_bits[-1] if history_none_bits else 0,
                   1)

    
    def show_total_view():
        nonlocal live_mode, active_run_id
        live_mode = False
        active_run_id = 0
    
        line_up.set_visible(True)
        line_down.set_visible(True)
        line_none.set_visible(True)

        line_live.set_data([], [])
        live_x.clear()
        live_y.clear()
        live_run_label.config(text="")
    
        # opdater data
        line_up.set_data(history_up_bits, history_up)
        line_down.set_data(history_down_bits, history_down)
        line_none.set_data(history_none_bits, history_none)
            
        # x-akse
        xmax = max_history_bits()
        ax.set_xlim(0, xmax)
    
        # ✅ sigma1 skal defineres inden den bruges
        sigma1 = math.sqrt(max(1, xmax))
    
        # y-akse baseret på historik men mindst ±1σ
        y_vals = []
        
        if history_up:   y_vals += history_up
        if history_down: y_vals += history_down
        if history_none: y_vals += history_none
    
        if y_vals:
            y_abs = max(max(abs(min(y_vals)), abs(max(y_vals))), sigma1)
        else:
            y_abs = sigma1
    
        pad = 0.08 * y_abs
        ax.set_ylim(-y_abs - pad, y_abs + pad)
    
        update_sigma_lines(xmax)    
        enforce_sigma_visibility()         # hvis du bruger den model
        refresh_legend()
        canvas.draw_idle()


        
    # ✅ KALD DEN HER (nu er den defineret)
    show_total_view()

    
    def gui_update(after_ms, func, *args, **kwargs):
        root.after(after_ms, lambda: func(*args, **kwargs))

    def on_close():
        global current_worker
        stop_event.set()
    
        # 1) Vent på evt. igangværende worker, så dens 'finally' kan sende STOP selv
        try:
            if current_worker and current_worker.is_alive():
                current_worker.join(timeout=TIMEOUT + 1.0)
        except Exception:
            pass
        finally:
            current_worker = None
    
        # 2) Hvis der ikke var en worker der lukkede pænt ned, så send STOP her
        try:
            with ser_lock:
                if ser and ser.is_open:
                    try:
                        ser.write(b"STOP\n")
                        ser.flush()
                        time.sleep(0.05)
                        ser.reset_input_buffer()
                        ser.reset_output_buffer()
                    except Exception:
                        pass
        except Exception:
            pass
    
        # 3) Luk log og serieport
        try:
            log_file.close()
        except Exception:
            pass
    
        try:
            with ser_lock:
                if ser and ser.is_open:
                    ser.close()
        except Exception:
            pass
    
        root.destroy()



    def update_stat_labels(last_val=None, last_cnt=None, last_dir=None):
        # Z/p for UP/DOWN (kumulativt)
        z_up   = z_from_accum(accum_up,   cum_up_bits)
        z_down = z_from_accum(accum_down, cum_down_bits)
        z_none = z_from_accum(accum_none, cum_none_bits)

    
        p_up_2   = p_two_sided_from_z(z_up)
        p_down_2 = p_two_sided_from_z(z_down)
        p_none_2 = p_two_sided_from_z(z_none)

    
        p_up_1   = p_one_sided_up(z_up)       # rettet mod +
        p_down_1 = p_one_sided_down(z_down)   # rettet mod −
    
        # Kombineret: Fisher (tosidet)
        p_comb_fisher  = fisher_p_from_two_sided(p_up_2, p_down_2)
        z_comb_fisher  = z_equiv_from_two_sided_p(p_comb_fisher)
    
        # Kombineret: Stouffer (retningsbevarende, vægtet med sqrt(N))
        z_stouf, p_stouf_1s, p_stouf_2s = stouffer_combined(
            z_up, cum_up_bits, z_down, cum_down_bits, directional=True
        )
    
        # Produkt af rettede 1s-p (kun som sanity)
        p_comb_product = p_up_1 * p_down_1 if (math.isfinite(p_up_1) and math.isfinite(p_down_1)) else float("nan")
    
        # Opdater faste felter
        up_acc_label.config(text=f"{accum_up:+d}")
        down_acc_label.config(text=f"{accum_down:+d}")
        none_acc_label.config(text=f"{accum_none:+d}")
        
        up_samples_label.config(text=f"{cum_up_raw:,} rå samples  |  {cum_up_bits:,} bits")
        down_samples_label.config(text=f"{cum_down_raw:,} rå samples  |  {cum_down_bits:,} bits")
        none_samples_label.config(text=f"{cum_none_raw:,} rå samples  |  {cum_none_bits:,} bits")

        p_up_label.config(
            text=f"Z={z_up:+.2f}  |  p≈{p_up_1:.3g} (1s) / {p_up_2:.3g} (2s)"
        )


        p_down_label.config(
            text=f"Z={z_down:+.2f}  |  p≈{p_down_1:.3g} (1s) / {p_down_2:.3g} (2s)"
        )

        p_comb_label.config(
            text=(
                f"Fisher: p≈{p_comb_fisher:.3g} (2s)   •   "
                f"Stouffer: Z={z_stouf:+.2f} | p≈{p_stouf_1s:.3g} (1s) / {p_stouf_2s:.3g} (2s)"
            )
        )



        p_none_label.config(text=f"Z={z_none:+.2f}  |  p≈{p_none_2:.3g} (2s)")
    
        if last_cnt is not None and last_cnt > 0:
            z_last  = z_from_accum(last_val, last_cnt)
            p_last2 = p_two_sided_from_z(z_last)
        
            m = last_run_mode or "?"
        
            # --- NONE / baseline run ---
            if last_dir == "NONE":
                p_last_label.config(
                    text=(
                        f"Sidste run ({m}, NO INTENTION)  N={last_cnt:,}  |  Δ={last_val:+.0f}  |  "
                        f"Z={z_last:+.2f}  |  p≈{p_last2:.3g} (2s)"
                    )
                )
                return
        
            # --- UP/DOWN run ---
            if last_dir in ("UP", "DOWN"):
                chosen_dir = last_dir
                if last_dir == "UP":
                    p_last1 = p_one_sided_up(z_last)
                else:
                    p_last1 = p_one_sided_down(z_last)
        
            # --- fallback hvis last_dir er ukendt ---
            else:
                p_up_last   = p_one_sided_up(z_last)
                p_down_last = p_one_sided_down(z_last)
        
                if p_up_last <= p_down_last:
                    chosen_dir = "UP"
                    p_last1 = p_up_last
                else:
                    chosen_dir = "DOWN"
                    p_last1 = p_down_last
        
            p_last_label.config(
                text=(
                    f"Sidste run ({m}, intention: {chosen_dir})  N={last_cnt:,}  |  Δ={last_val:+.0f}  |  "
                    f"Z={z_last:+.2f}  |  p≈{p_last1:.3g} (1s) / {p_last2:.3g} (2s)"
                )
            )


    def do_run(direction, mode, this_run_id):
        
        TARGET_VN_BITS = 1_000_000
        total_to_read  = 12_000_000   # failsafe (kan være 8-20 mio)

        gui_update(0, set_ui_running, True, direction)

        global global_toggle
        thread_name = current_thread().name
        dbg("RUN_BEGIN", dir=direction, thread=thread_name)

        nonlocal accum_up, accum_down, accum_none
        nonlocal cum_up_raw, cum_down_raw, cum_none_raw
        nonlocal cum_up_bits, cum_down_bits, cum_none_bits
        nonlocal history_up, history_down, history_none
        nonlocal history_up_bits, history_down_bits, history_none_bits
        
        nonlocal last_run_dir, last_run_val, last_run_bits, last_run_mode

        nonlocal active_run_id


        try:
            # Gør knapperne inaktive og giv visuel feedback
            

            #if direction == "UP":
            #    gui_update(0, btn_up.config, state=tk.DISABLED, bg="#a8e6a3", activebackground="#a8e6a3")
            #    gui_update(0, btn_down.config, state=tk.DISABLED, bg="#f0f0f0")
            #    gui_update(0, btn_quit.config, state=tk.DISABLED)
    
                
            #else:
            #    gui_update(0, btn_down.config, state=tk.DISABLED, bg="#a3b4e6", activebackground="#a3b4e6")
            #    gui_update(0, btn_up.config, state=tk.DISABLED, bg="#f0f0f0")
            #    gui_update(0, btn_quit.config, state=tk.DISABLED)
   

            total_to_read = 600 * SAMPLES_PER_SECOND
            print(f">> Henter {total_to_read} samples...")
            
            dbg("RUN_CONF", total_samples=total_to_read, ds=DOWNSAMPLE)


            def progress_cb(accum_val, bit_cnt, raw_used, _dir=direction):
                gui_update(0, update_live_run, _dir, accum_val, bit_cnt, raw_used, this_run_id)



            val, cnt, raw_cnt, bits_str, global_toggle = record_samples(
                ser,
                total_samples=total_to_read,
                return_bits=True,
                toggle_start=global_toggle,
                on_progress=progress_cb,
                target_vn_bits=TARGET_VN_BITS
            )

            
            # Gem bit-dump
            ts_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            dump_name = f"run_bits_{direction}_{ts_stamp}.txt"
            dump_path = os.path.join(BIT_DUMP_DIR, dump_name)
            try:
                with open(dump_path, "w", encoding="utf-8") as f:
                    f.write(bits_str)
                print(f"[BIT-DUMP] Skrev {len(bits_str)} bits til {dump_path}")
            except Exception as e:
                print("Kunne ikke skrive bit-dump:", repr(e))



            dbg("RUN_GOT", val=f"{val:.4f}", bits=cnt, raw=raw_cnt)
            
            if stop_event.is_set():
                return

            # Gem LAST RUN-info
            last_run_dir  = direction
            last_run_val  = val
            last_run_bits = cnt
            last_run_mode = mode

            if direction == 'UP':
                accum_up += val
                cum_up_bits += cnt
                cum_up_raw += raw_cnt
                history_up.append(accum_up)
                history_up_bits.append(cum_up_bits)
            
            elif direction == 'DOWN':
                accum_down += val
                cum_down_bits += cnt
                cum_down_raw += raw_cnt
                history_down.append(accum_down)
                history_down_bits.append(cum_down_bits)
            
            else:  # NONE
                accum_none += val
                cum_none_bits += cnt
                cum_none_raw += raw_cnt
                history_none.append(accum_none)
                history_none_bits.append(cum_none_bits)
            

            # Log
            ts = datetime.now().isoformat()
            if direction == 'UP':
                total = accum_up
            elif direction == 'DOWN':
                total = accum_down
            else:
                total = accum_none

            raw_samples_used = raw_cnt
            try:
                log_file.write(f"{ts},{mode},{direction},{int(total)},{cnt},{raw_samples_used}\n")
                log_file.flush()
            except Exception as e:
                print("Logfejl:", repr(e))

            def apply_updates():
                nonlocal live_mode, active_run_id
            
                live_mode = False
                active_run_id = 0
            
                update_stat_labels(last_val=last_run_val, last_cnt=last_run_bits, last_dir=last_run_dir)
            
                show_total_view()  # <-- denne alene

                set_ui_running(False)


            dbg("RUN_UPDATE_STATE",
                accum_up=f"{accum_up:.4f}", bits_up=cum_up_bits,
                accum_down=f"{accum_down:.4f}", bits_down=cum_down_bits, dir=direction)

            gui_update(0, apply_updates)
            nonlocal last_done_time
            last_done_time = time.monotonic()
            dbg("RUN_DONE_SCHEDULED", dt=f"{last_done_time-last_start_time:.3f}s")

        except Exception as e:
            nonlocal last_err
            last_err = repr(e)
            dbg("WORKER_ERR", err=last_err)
            traceback.print_exc()
        
            # GUI-reset fra worker-tråd
            gui_update(0, set_ui_running, False, None)
            
        finally:
            # Markér at tråden er færdig, så nye klik kan starte næste run
            global current_worker
            current_worker = None
            dbg("THREAD_END")

    def start_run(dirn, mode="MANUAL"):

        global current_worker
        nonlocal last_click_time, last_start_time, run_seq
        
        nonlocal active_run_id
        nonlocal live_mode, run_target_bits, run_target_raw
        nonlocal run_start_monotonic
        
        now = time.monotonic()
        dbg("CLICK", dir=dirn, since_last=f"{now-last_click_time:.3f}s")
        last_click_time = now
    
        # Debounce: ignorer klik der lander for tæt på
        if now - last_done_time < MIN_CLICK_GAP:
            dbg("CLICK_IGNORED_DEBOUNCE", gap=f"{now-last_done_time:.3f}s")
            return
    
        # Undgå samtidige runs
        if current_worker and current_worker.is_alive():
            dbg("CLICK_IGNORED_BUSY")
            return
    
        stop_event.clear()
        run_seq += 1
        
        active_run_id = run_seq
        this_run_id = active_run_id

        # --- LIVE VIEW SETUP ---

        live_mode = True

        # Forventet antal bits i dette run (samme som du bruger i do_run)
        TARGET_VN_BITS = 1_000_000
        
        run_target_bits = TARGET_VN_BITS
        run_target_raw = 600 * SAMPLES_PER_SECOND   # bare failsafe-grænse

        # Nulstil live-kurve
        live_x.clear()
        live_y.clear()
        line_live.set_data([], [])

        # Skjul historik mens der køres, så x-skalaen ikke domineres af historikken
        line_up.set_visible(False)
        line_down.set_visible(False)
        line_none.set_visible(False)


        # Lås x med det samme (så den fylder hele bredden fra start)
        ax.set_xlim(0, run_target_bits)
        
        sigma1 = math.sqrt(max(1, run_target_bits))
        ax.set_ylim(-sigma1, sigma1)
        
        canvas.draw_idle()
        
        run_start_monotonic = time.monotonic()
        
        t = threading.Thread(target=do_run, args=(dirn, mode, this_run_id), daemon=True, name=f"worker#{run_seq}")
        current_worker = t
        last_start_time = now

        t.start()
        

    # --- UI state (én sandhed) ---
    run_in_progress = False
    
    def set_ui_running(is_running: bool, direction=None):
        nonlocal run_in_progress
        run_in_progress = is_running
    
        if is_running:
            # Disable alt mens der køres
            btn_up.config(state=tk.DISABLED)
            btn_down.config(state=tk.DISABLED)
            btn_none.config(state=tk.DISABLED)
            btn_quit.config(state=tk.DISABLED)
            btn_rand.config(state=tk.DISABLED)


            # Vis hvilken retning der kører (farve kan stadig bruges)
            btn_up.config(bg="SystemButtonFace", activebackground="SystemButtonFace")
            btn_down.config(bg="SystemButtonFace", activebackground="SystemButtonFace")
            btn_none.config(bg="SystemButtonFace", activebackground="SystemButtonFace")

            if direction == "UP":
                btn_up.config(bg="#a8e6a3", activebackground="#a8e6a3")
            elif direction == "DOWN":
                btn_down.config(bg="#a3b4e6", activebackground="#a3b4e6")
            elif direction == "NONE":
                btn_none.config(bg="#d9d9d9", activebackground="#d9d9d9")


        else:
            # Enable igen
            btn_up.config(state=tk.NORMAL, bg="SystemButtonFace", activebackground="SystemButtonFace")
            btn_down.config(state=tk.NORMAL, bg="SystemButtonFace", activebackground="SystemButtonFace")
            btn_quit.config(state=tk.NORMAL)
            btn_none.config(state=tk.NORMAL, bg="SystemButtonFace", activebackground="SystemButtonFace")
            btn_rand.config(state=tk.NORMAL)
            
    def try_close():
        if run_in_progress:
            messagebox.showwarning(
                "Measurement in progress",
                "A measurement is currently running.\n\n"
                "Please wait until it has finished,\n"
            )
            return
    
        on_close()
        
    def start_random_intention():
        # kun UP/DOWN så intention giver mening
        choice = random.choice(["UP", "DOWN"])
        print(f">> RANDOM valgt: {choice}")  # vises før run'et
        start_run(choice, mode="RANDOM")

    
    root.protocol("WM_DELETE_WINDOW", try_close)

    # knapper
    btn_frame = tk.Frame(root)
    btn_frame.pack(pady=(4,2))
        
    btn_up    = tk.Button(btn_frame, text="Up",           width=10, command=lambda: start_run('UP',   mode="MANUAL"))
    btn_down  = tk.Button(btn_frame, text="Down",         width=10, command=lambda: start_run('DOWN', mode="MANUAL"))
    btn_rand  = tk.Button(btn_frame, text="Random",       width=10, command=start_random_intention)
    btn_none  = tk.Button(btn_frame, text="No Intention", width=12, command=lambda: start_run('NONE', mode="MANUAL"))
    btn_quit  = tk.Button(btn_frame, text="Quit",         width=10, command=try_close)
    
    btn_up.grid(row=0, column=0, padx=4)
    btn_down.grid(row=0, column=1, padx=4)
    btn_rand.grid(row=0, column=2, padx=4)
    btn_none.grid(row=0, column=3, padx=4)
    btn_quit.grid(row=0, column=4, padx=4)


    # initiale labels
    update_stat_labels()

    #root.protocol("WM_DELETE_WINDOW", on_close)
    root.mainloop()

if __name__ == "__main__":
    main()